In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pydicom as dicom
import sys
import matplotlib.path as mplPath
import pandas as pd
import shutil
import os
import glob
import csv
from skimage import io

import torch
from torchvision import datasets
import torch.nn as nn
from torch.utils.data import Dataset
from torchvision.io import read_image
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader,WeightedRandomSampler

# defining transformation
# data augmentation pipeline
import torchvision.transforms as transforms
from torchvision import datasets

train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.RandomRotation(50),
    transforms.GaussianBlur(kernel_size = (3, 3), sigma = (0.3, 0.7)),
    transforms.RandomVerticalFlip(),
    transforms.RandomAffine(degrees=0, translate=(0.035, 0.035)),
    transforms.Resize((224, 224))
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((224, 224))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((224, 224))
])

trainset = datasets.ImageFolder(root = 'Splitted2D/train', transform = train_transform)
valset = datasets.ImageFolder(root = 'Splitted2D/val', transform = val_transform)
testset = datasets.ImageFolder(root = 'Splitted2D/test', transform = test_transform)

idx2class_train = {v: k for k, v in trainset.class_to_idx.items()}
idex2class_val = {v: k for k, v in valset.class_to_idx.items()}
idex2class_test = {v: k for k, v in testset.class_to_idx.items()}

def get_class_distribution(dataset_obj, idx2class):
    count_dict = {k:0 for k,v in dataset_obj.class_to_idx.items()}
    
    for element in dataset_obj:
        y_lbl = element[1]
        y_lbl = idx2class[y_lbl]
        count_dict[y_lbl] += 1
            
    return count_dict
#print("Distribution of classes: \n", get_class_distribution(trainset))


train_list = torch.tensor(trainset.targets)
class_count_train = [i for i in get_class_distribution(trainset, idx2class_train).values()]
train_weights = 1./torch.tensor(class_count_train, dtype=torch.float) 
train_weights_all = train_weights[train_list]

val_list = torch.tensor(valset.targets)
class_count_val = [i for i in get_class_distribution(valset, idx2class_train).values()]
val_weights = 1./torch.tensor(class_count_val, dtype = torch.float)
val_weights_all = val_weights[val_list]

test_list = torch.tensor(testset.targets)
class_count_test = [i for i in get_class_distribution(testset, idx2class_train).values()]
test_weights = 1./torch.tensor(class_count_test, dtype = torch.float)
test_weights_all = test_weights[test_list]

train_weights_all

weighted_train_sampler = WeightedRandomSampler(
    weights=train_weights_all,
    num_samples=len(train_weights_all),
    replacement=True
)
weighted_val_sampler = WeightedRandomSampler(
    weights = val_weights_all,
    num_samples = len(val_weights_all),
    replacement = True
)
weighted_test_sampler = WeightedRandomSampler(
    weights = test_weights_all,
    num_samples = len(test_weights_all),
    replacement = True
)

batch_size = 16
train_loader = DataLoader(trainset, batch_size = batch_size, sampler = weighted_train_sampler)
val_loader = DataLoader(valset, batch_size = batch_size, shuffle = True)
test_loader = DataLoader(testset, batch_size = batch_size, shuffle = True)

In [ ]:
from torchvision import models

num_epochs = 40
lr = 0.01
num_class = 4
model = models.vit_b_16(weights='IMAGENET1K_V1')
model.heads = nn.Sequential(
    #nn.Dropout(0.3, inplace = False),
    nn.Linear(768, 4),
    #nn.Softmax(dim = -1)
)
model.train()
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
loss = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr= 0.01) 
model = model.to(device)

In [ ]:
model

In [ ]:
from torchsummary import summary
summary(model, (3, 224, 224))

In [ ]:
def train_loop(train_loader, val_loader, model, loss_fn, optimizer):
    size = len(train_loader.dataset)
    correct = 0
    samples = 0
    
    for batch, (X, y) in enumerate(train_loader):
        # Compute prediction and loss
        X, y = X.to(device), y.to(device)
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        optimizer.zero_grad()
#         for name, param in model.named_parameters():
#             if param.grad is not None:
#                 gradient_norm = torch.norm(param.grad)
#                print(gradient_norm)
        loss.backward()
        optimizer.step()
        
        with torch.no_grad():
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
            samples += pred.size(0)
            if batch % 5 ==0:
                print(f"Acc: {float(correct) / float(samples) * 100:.2f}% Out of {samples} \n")
                correct = 0
                samples = 0

        if batch % 5 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]  \n")
    val_correct, val_samples = 0, 0
    for x, y in val_loader:
        x, y = x.to(device), y.to(device)
        pred = model(x)
        with torch.no_grad():
            val_correct += (pred.argmax(1) == y).type(torch.float).sum().item()
            val_samples += pred.size(0)
    print(f"validation accuracy: {val_correct / val_samples * 100:.2f}%")

def test_loop(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}% out of {size}, Avg loss: {test_loss:>8f} \n")

In [ ]:
num_epochs = 40
for t in range(num_epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_loader, val_loader, model, loss, optimizer)
